# ASMSA v2 — protein+ligand complex AAE

This notebook demonstrates the primary use case: training an AAE on a trajectory of a **complex** (protein + ligand, or two proteins). The feature vector per frame combines:

- **Intra-component** features (bonds / angles / dihedrals inside each component) — same as v1
- **Inter-component** distances between the two components — the new signal that encodes the binding pose

For speed, the notebook synthesizes 3-D coordinates of a toy complex (a rigid 'protein' cluster and a 'ligand' whose position relative to the protein changes along the trajectory). To run on a real MD trajectory, replace cell 2 with mdtraj + a real PDB — see the markdown cell before the training step for the drop-in snippet.

In [ ]:
import math
import numpy as np
import torch
from torch.utils.data import DataLoader

from asmsa_v2 import (
    AAE, Trainer, TrainConfig, IsotropicNormalPrior,
    ComplexMolecule, MultiMolDataset, multi_mol_collate,
    featurize_frames, set_seed,
)
from asmsa_v2.viz.latent import scatter_latents

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 1. Synthesize a toy complex trajectory

- **Protein**: 20 atoms (indices 0..19), fluctuating around a rigid cluster.
- **Ligand**: 5 atoms (indices 20..24), orbiting the protein's binding site in two distinct 'poses' (two latent modes).
- 2000 frames, 3-D coordinates in nm.

In a real workflow this cell is replaced with ``traj = md.load('traj.xtc', top='complex.pdb'); geoms = traj.xyz`` plus a small amount of atom-index bookkeeping.

In [ ]:
N_PROT, N_LIG, N_FRAMES = 20, 5, 2000
N_ATOMS = N_PROT + N_LIG
rng = np.random.default_rng(0)

# Rigid protein scaffold (position-fixed) + small jitter per frame
protein_scaffold = rng.standard_normal((N_PROT, 3)).astype(np.float32)
protein_scaffold -= protein_scaffold.mean(axis=0, keepdims=True)
protein_frames = (protein_scaffold[:, :, None]
                  + 0.05 * rng.standard_normal((N_PROT, 3, N_FRAMES)).astype(np.float32))

# Ligand: two binding poses, 'bound' at (3, 0, 0), 'unbound' at (0, 3, 2)
ligand_scaffold = 0.4 * rng.standard_normal((N_LIG, 3)).astype(np.float32)
pose_centers = np.stack([np.array([3., 0., 0.]), np.array([0., 3., 2.])])  # (2, 3)
pose_id = rng.integers(0, 2, size=N_FRAMES)
ligand_frames = (ligand_scaffold[:, :, None]
                 + pose_centers[pose_id].T[None, :, :]
                 + 0.1 * rng.standard_normal((N_LIG, 3, N_FRAMES)).astype(np.float32))

geoms = np.concatenate([protein_frames, ligand_frames], axis=0)  # [N_ATOMS, 3, N_FRAMES]
print('geoms shape:', geoms.shape)

## 2. Featurize with `ComplexMolecule`

- intra-protein features: a handful of pseudo-bonds (nearest-neighbor pairs in the scaffold)
- intra-ligand features: all intra-ligand pairwise distances
- inter-component features: dense (every protein × ligand pair = 20 × 5 = 100 distances)

In [ ]:
# Pseudo-bonds for intra features (just adjacent atoms in the scaffolds)
protein_bonds = np.array([[i, i + 1] for i in range(N_PROT - 1)], dtype=np.int32)
ligand_bonds = np.array([[i, j] for i in range(N_PROT, N_ATOMS)
                                  for j in range(i + 1, N_ATOMS)], dtype=np.int32)
intra_bonds = np.vstack([protein_bonds, ligand_bonds])

cm = ComplexMolecule(
    n_atoms=N_ATOMS,
    components={'protein': range(N_PROT), 'ligand': range(N_PROT, N_ATOMS)},
    bonds=intra_bonds,
    inter_mode='dense',
)
mol_model = cm.get_model()

# Featurize all frames: [N_FRAMES, n_features]
X = featurize_frames(np.transpose(geoms, (2, 0, 1)), mol_model)  # featurize_frames expects [n_frames, n_atoms, 3]
print('feature tensor:', X.shape, '(each row = 1 frame)')

**With a real PDB** replace cells 1-2 with:

```python
import mdtraj as md
traj = md.load('traj.xtc', top='complex.pdb')
protein_atoms = traj.top.select('protein')
ligand_atoms  = traj.top.select('resname LIG')
cm = ComplexMolecule(
    pdb='complex.pdb', n_atoms=traj.n_atoms,
    components={'protein': protein_atoms, 'ligand': ligand_atoms},
    diheds=my_dihed_indices,           # or extract via asmsa_v2.extract_dihedral_indices_local
    inter_subsets={'protein': traj.top.select('name CA'),
                   'ligand':  traj.top.select('resname LIG and not symbol H')},
    inter_mode='dense',
)
X = featurize_frames(traj.xyz, cm.get_model())
```

## 3. Train the AAE

In [ ]:
ds = MultiMolDataset({'complex': X})  # single-complex case
loader = DataLoader(ds,
                    batch_sampler=ds.round_robin_sampler(batch_size=128),
                    collate_fn=multi_mol_collate)

model = AAE(feature_dims=ds.feature_dims, latent_dim=2,
            encoder_widths=(128, 64, 32), decoder_widths=(32, 64, 128),
            disc_widths=(64, 64))

cfg = TrainConfig(epochs=30, batch_size=128,
                  lr_ae=2e-4, lr_d=1e-4, lr_g=2e-4, device=device)
trainer = Trainer(model, prior=IsotropicNormalPrior(2), config=cfg)
history = trainer.fit(loader)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
for k, v in history.items():
    ax.plot(v, label=k)
ax.axhline(math.log(2) * 2, color='k', ls='--', lw=0.8, label='2·log 2')
ax.set_xlabel('epoch'); ax.set_yscale('log'); ax.legend()
plt.tight_layout(); plt.show()

## 4. Latent space — the two binding poses should separate

Color-code frames by the ground-truth pose index. If training worked, the two poses occupy distinct regions of the latent.

In [ ]:
latents = trainer.encode_all({'complex': X})
z = latents['complex'].numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(z[pose_id == 0, 0], z[pose_id == 0, 1], s=6, alpha=0.5, label='pose 0')
ax.scatter(z[pose_id == 1, 0], z[pose_id == 1, 1], s=6, alpha=0.5, label='pose 1')
ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]'); ax.legend()
ax.set_aspect('equal', adjustable='datalim')
plt.tight_layout(); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    X_dev = X.to(trainer.device)
    x_hat, _ = model(X_dev, 'complex')
    mse = torch.mean((x_hat - X_dev) ** 2).item()
print(f'reconstruction MSE on the full trajectory: {mse:.4f}')